# Predictive Analytics for Employee Attrition
### A Comparative Study of Machine Learning Algorithms

Implementation of the methodology in *Krishna, Dwivedi & Murti (2023), DDP* on the
IBM HR Employee Attrition dataset (1470 rows, 35 columns).

Models, exactly as in the paper:
- **Random Forest** (all features, selected features, + grid search)
- **K-Nearest Neighbours (K = 5)** (all features, selected features, + grid search)
- **Naive Bayes** (all features, selected features)
- **Two weighted ensembles** (soft voting) combining NB + K-NN + RF

Feature engineering as in the paper: one-hot encoding of categoricals, Random-Forest
feature-importance based selection, and feature scaling (StandardScaler).
Evaluation metrics: Accuracy and F1 score, plus confusion matrices.

> Dataset is **assumed clean** (no missing values, duplicates, or outliers), so no
> imputation / cleaning is performed. No fixed random seed is set, so accuracies will
> vary slightly between runs around the paper's reported figures.


## 1. Imports

In [17]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

pd.set_option("display.max_columns", None)

## 2. Load the data


In [18]:
DATA_PATH = "/kaggle/input/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv"

dataset = pd.read_csv(DATA_PATH)
print("Shape:", dataset.shape)        # (1470, 35)
dataset.head()

Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## 3. Data type conversion (target + one-hot encoding)

The target `Attrition` is separated as `y`; the remaining columns form `X`.
Categorical inputs are converted to a numerical representation with one-hot
encoding (pandas), as in the paper.

In [19]:
# Target: Yes -> 1 (attrition), No -> 0 (no attrition)
y = dataset["Attrition"].map({"No": 0, "Yes": 1}).astype(int)

# Features
X = dataset.drop(columns=["Attrition"])

# Convert categorical columns to numerical via one-hot encoding (pandas)
categorical_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols)
X = X.astype(float)

print("Categorical columns encoded:", categorical_cols)
print("Feature matrix shape after encoding:", X.shape)

# Attrition distribution (the dataset is imbalanced: ~16% attrition)
print("\nClass distribution:\n", y.value_counts(normalize=True).round(3))

Categorical columns encoded: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime']
Feature matrix shape after encoding: (1470, 55)

Class distribution:
 Attrition
0    0.839
1    0.161
Name: proportion, dtype: float64


## 4. Train / test split

A 70/30 split is used (test size 0.30 → 441 test instances, matching the
confusion-matrix totals reported in the paper).

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 1029 | Test: 441


## 5. Random Forest — all features

Random Forest is an ensemble of decision trees. We first build it on the complete
feature set and inspect training accuracy and the confusion matrix (Fig. 2 in the
paper). The paper reports a training accuracy of roughly **86%**.

In [21]:
rf_all = RandomForestClassifier()
rf_all.fit(X_train, y_train)

print("RF training accuracy: {:.4f}".format(rf_all.score(X_train, y_train)))

y_pred_rf_all = rf_all.predict(X_test)
print("RF test accuracy:     {:.4f}".format(accuracy_score(y_test, y_pred_rf_all)))

print("\nConfusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, y_pred_rf_all))
print("\n", classification_report(y_test, y_pred_rf_all,
                                    target_names=["No Attrition", "Attrition"]))

RF training accuracy: 1.0000
RF test accuracy:     0.8707

Confusion matrix (rows = actual, cols = predicted):
[[369   1]
 [ 56  15]]

               precision    recall  f1-score   support

No Attrition       0.87      1.00      0.93       370
   Attrition       0.94      0.21      0.34        71

    accuracy                           0.87       441
   macro avg       0.90      0.60      0.64       441
weighted avg       0.88      0.87      0.83       441



## 6. Feature selection via Random-Forest importances

The relative importance of each feature is read off the fitted Random Forest
(Fig. 3 in the paper). Two feature sets are derived from these importances:

- **Selected Features** — features with relative importance ≥ 0.01
- **New Selected Features** — a tighter, refined subset (relative importance ≥ 0.02)

In [22]:
importances = (
    pd.Series(rf_all.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
)
importances.head(20)

MonthlyIncome              0.066521
Age                        0.053695
MonthlyRate                0.052307
DailyRate                  0.050522
DistanceFromHome           0.046668
EmployeeNumber             0.043283
TotalWorkingYears          0.040987
HourlyRate                 0.037993
YearsWithCurrManager       0.034609
YearsAtCompany             0.034444
EnvironmentSatisfaction    0.031322
PercentSalaryHike          0.029103
YearsInCurrentRole         0.027995
NumCompaniesWorked         0.027788
TrainingTimesLastYear      0.026978
StockOptionLevel           0.026140
JobSatisfaction            0.025438
OverTime_No                0.025093
YearsSinceLastPromotion    0.024981
WorkLifeBalance            0.023927
dtype: float64

In [23]:
# Selected features (relative importance >= 0.01)
selected_features = importances[importances >= 0.01].index.tolist()
if len(selected_features) < 5:                 # safety fallback
    selected_features = importances.head(15).index.tolist()

# New (refined) selected features (relative importance >= 0.02)
new_selected_features = importances[importances >= 0.02].index.tolist()
if len(new_selected_features) < 5:             # safety fallback
    new_selected_features = importances.head(8).index.tolist()

print("Selected Features ({}):".format(len(selected_features)))
print(selected_features)
print("\nNew Selected Features ({}):".format(len(new_selected_features)))
print(new_selected_features)

Selected Features (27):
['MonthlyIncome', 'Age', 'MonthlyRate', 'DailyRate', 'DistanceFromHome', 'EmployeeNumber', 'TotalWorkingYears', 'HourlyRate', 'YearsWithCurrManager', 'YearsAtCompany', 'EnvironmentSatisfaction', 'PercentSalaryHike', 'YearsInCurrentRole', 'NumCompaniesWorked', 'TrainingTimesLastYear', 'StockOptionLevel', 'JobSatisfaction', 'OverTime_No', 'YearsSinceLastPromotion', 'WorkLifeBalance', 'JobInvolvement', 'JobLevel', 'OverTime_Yes', 'RelationshipSatisfaction', 'Education', 'MaritalStatus_Single', 'BusinessTravel_Travel_Frequently']

New Selected Features (24):
['MonthlyIncome', 'Age', 'MonthlyRate', 'DailyRate', 'DistanceFromHome', 'EmployeeNumber', 'TotalWorkingYears', 'HourlyRate', 'YearsWithCurrManager', 'YearsAtCompany', 'EnvironmentSatisfaction', 'PercentSalaryHike', 'YearsInCurrentRole', 'NumCompaniesWorked', 'TrainingTimesLastYear', 'StockOptionLevel', 'JobSatisfaction', 'OverTime_No', 'YearsSinceLastPromotion', 'WorkLifeBalance', 'JobInvolvement', 'JobLevel', 

## 7. Feature scaling helper

Feature scaling normalises the range of the variables, which matters for distance-
and probability-based learners (K-NN). A small helper returns the train/test slices
for a given feature set, optionally StandardScaler-normalised.

In [24]:
def make_sets(features, scale=False):
    """Return (X_train_fs, X_test_fs) for the given feature list."""
    Xtr = X_train[features].copy()
    Xte = X_test[features].copy()
    if scale:
        sc = StandardScaler()
        Xtr = pd.DataFrame(sc.fit_transform(Xtr), columns=features, index=Xtr.index)
        Xte = pd.DataFrame(sc.transform(Xte),    columns=features, index=Xte.index)
    return Xtr, Xte


def evaluate(model, Xte, yte, name):
    """Fit-already model -> print accuracy, F1, confusion matrix; return (acc, f1)."""
    pred = model.predict(Xte)
    acc = accuracy_score(yte, pred)
    f1  = f1_score(yte, pred)
    print("{:<45} Accuracy = {:.4%}  F1 = {:.4%}".format(name, acc, f1))
    print("   Confusion matrix:", confusion_matrix(yte, pred).tolist())
    return acc, f1

results = {}   # collected for the final summary table

## 8. Random Forest — selected vs. new selected features

The RF classifier is re-evaluated on the two reduced feature sets, then tuned with
`GridSearchCV` over `n_estimators`, `max_features`, `max_depth`, `min_samples_split`,
`min_samples_leaf`, and `bootstrap`, scored on F1 (the paper's chosen metric).

In [25]:
# --- RF on Selected Features (no grid search) ---
Xtr_s, Xte_s = make_sets(selected_features)
rf_sel = RandomForestClassifier().fit(Xtr_s, y_train)
results[("RF", "Selected")] = evaluate(rf_sel, Xte_s, y_test, "RF (Selected Features)")

# --- RF on New Selected Features (no grid search) ---
Xtr_ns, Xte_ns = make_sets(new_selected_features)
rf_newsel = RandomForestClassifier().fit(Xtr_ns, y_train)
results[("RF", "New Selected")] = evaluate(rf_newsel, Xte_ns, y_test,
                                           "RF (New Selected Features)")

RF (Selected Features)                        Accuracy = 85.7143%  F1 = 30.7692%
   Confusion matrix: [[364, 6], [57, 14]]
RF (New Selected Features)                    Accuracy = 86.3946%  F1 = 34.7826%
   Confusion matrix: [[365, 5], [55, 16]]


In [26]:
rf_param_grid = {
    "n_estimators":      [100, 200],
    "max_features":      ["sqrt", "log2"],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2],
    "bootstrap":         [True, False],
}

# Grid search RF on ALL features (used later in Ensemble 1)
rf_grid_all = GridSearchCV(RandomForestClassifier(), rf_param_grid,
                           scoring="f1", cv=3, n_jobs=-1)
rf_grid_all.fit(X_train, y_train)
print("Best RF params (all features):", rf_grid_all.best_params_)
evaluate(rf_grid_all.best_estimator_, X_test, y_test, "RF (grid search + all features)")

# Grid search RF on SELECTED features
rf_grid_sel = GridSearchCV(RandomForestClassifier(), rf_param_grid,
                           scoring="f1", cv=3, n_jobs=-1)
rf_grid_sel.fit(Xtr_s, y_train)
print("\nBest RF params (selected):", rf_grid_sel.best_params_)
results[("RF", "Grid+Selected")] = evaluate(rf_grid_sel.best_estimator_, Xte_s, y_test,
                                            "RF (grid search + selected features)")

# Grid search RF on NEW SELECTED features
rf_grid_newsel = GridSearchCV(RandomForestClassifier(), rf_param_grid,
                              scoring="f1", cv=3, n_jobs=-1)
rf_grid_newsel.fit(Xtr_ns, y_train)
print("\nBest RF params (new selected):", rf_grid_newsel.best_params_)
results[("RF", "Grid+New Selected")] = evaluate(rf_grid_newsel.best_estimator_, Xte_ns,
                                                y_test, "RF (grid search + new selected)")

Best RF params (all features): {'bootstrap': False, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
RF (grid search + all features)               Accuracy = 87.5283%  F1 = 38.2022%
   Confusion matrix: [[369, 1], [54, 17]]

Best RF params (selected): {'bootstrap': False, 'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
RF (grid search + selected features)          Accuracy = 86.3946%  F1 = 36.1702%
   Confusion matrix: [[364, 6], [54, 17]]

Best RF params (new selected): {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
RF (grid search + new selected)               Accuracy = 86.6213%  F1 = 39.1753%
   Confusion matrix: [[363, 7], [52, 19]]


## 9. K-Nearest Neighbours

K-NN is a non-parametric "lazy" learner that classifies a point by its K nearest
neighbours (Euclidean distance by default). Scaled features are used. The base model
uses **K = 5**, then `GridSearchCV` tunes `n_neighbors`, `weights`, `algorithm`,
`metric`, and `p`, scored on F1.

In [27]:
# --- K-NN (K=5) on Selected Features ---
Xtr_s_sc, Xte_s_sc = make_sets(selected_features, scale=True)
knn_sel = KNeighborsClassifier(n_neighbors=5).fit(Xtr_s_sc, y_train)
results[("KNN", "Selected")] = evaluate(knn_sel, Xte_s_sc, y_test, "KNN (Selected Features)")

# --- K-NN (K=5) on New Selected Features ---
Xtr_ns_sc, Xte_ns_sc = make_sets(new_selected_features, scale=True)
knn_newsel = KNeighborsClassifier(n_neighbors=5).fit(Xtr_ns_sc, y_train)
results[("KNN", "New Selected")] = evaluate(knn_newsel, Xte_ns_sc, y_test,
                                            "KNN (New Selected Features)")

KNN (Selected Features)                       Accuracy = 83.6735%  F1 = 25.0000%
   Confusion matrix: [[357, 13], [59, 12]]
KNN (New Selected Features)                   Accuracy = 85.4875%  F1 = 31.9149%
   Confusion matrix: [[362, 8], [56, 15]]


In [28]:
knn_param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights":     ["uniform", "distance"],
    "algorithm":   ["auto", "ball_tree", "kd_tree"],
    "metric":      ["euclidean", "manhattan", "minkowski"],
    "p":           [1, 2],
}

# Grid search K-NN on SELECTED features
knn_grid_sel = GridSearchCV(KNeighborsClassifier(), knn_param_grid,
                            scoring="f1", cv=3, n_jobs=-1)
knn_grid_sel.fit(Xtr_s_sc, y_train)
print("Best K-NN params (selected):", knn_grid_sel.best_params_)
results[("KNN", "Grid+Selected")] = evaluate(knn_grid_sel.best_estimator_, Xte_s_sc,
                                             y_test, "KNN (grid search + selected)")

# Grid search K-NN on NEW SELECTED features
knn_grid_newsel = GridSearchCV(KNeighborsClassifier(), knn_param_grid,
                               scoring="f1", cv=3, n_jobs=-1)
knn_grid_newsel.fit(Xtr_ns_sc, y_train)
print("\nBest K-NN params (new selected):", knn_grid_newsel.best_params_)
results[("KNN", "Grid+New Selected")] = evaluate(knn_grid_newsel.best_estimator_,
                                                 Xte_ns_sc, y_test,
                                                 "KNN (grid search + new selected)")

Best K-NN params (selected): {'algorithm': 'auto', 'metric': 'euclidean', 'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}
KNN (grid search + selected)                  Accuracy = 83.9002%  F1 = 29.7030%
   Confusion matrix: [[355, 15], [56, 15]]

Best K-NN params (new selected): {'algorithm': 'auto', 'metric': 'manhattan', 'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}
KNN (grid search + new selected)              Accuracy = 85.7143%  F1 = 35.0515%
   Confusion matrix: [[361, 9], [54, 17]]


## 10. Naive Bayes

Gaussian Naive Bayes assumes feature independence. It is simple and does not benefit
much from hyperparameter tuning, but the paper finds it the strongest at **forecasting
attrition specifically**. The paper reports ~**75%** overall accuracy on all features
and a high attrition (recall) performance on the selected set.

In [29]:
# --- Naive Bayes on ALL features ---
nb_all = GaussianNB().fit(X_train, y_train)
acc_nb_all = accuracy_score(y_test, nb_all.predict(X_test))
print("NB (All Features) accuracy: {:.4%}".format(acc_nb_all))
print(classification_report(y_test, nb_all.predict(X_test),
                            target_names=["No Attrition", "Attrition"]))

# --- Naive Bayes on Selected Features ---
nb_sel = GaussianNB().fit(Xtr_s, y_train)
results[("NB", "Selected")] = evaluate(nb_sel, Xte_s, y_test, "NB (Selected Features)")

# --- Naive Bayes on New Selected Features ---
nb_newsel = GaussianNB().fit(Xtr_ns, y_train)
results[("NB", "New Selected")] = evaluate(nb_newsel, Xte_ns, y_test,
                                           "NB (New Selected Features)")

NB (All Features) accuracy: 74.3764%
              precision    recall  f1-score   support

No Attrition       0.93      0.75      0.83       370
   Attrition       0.35      0.69      0.46        71

    accuracy                           0.74       441
   macro avg       0.64      0.72      0.65       441
weighted avg       0.83      0.74      0.77       441

NB (Selected Features)                        Accuracy = 78.6848%  F1 = 50.5263%
   Confusion matrix: [[299, 71], [23, 48]]
NB (New Selected Features)                    Accuracy = 77.5510%  F1 = 48.1675%
   Confusion matrix: [[296, 74], [25, 46]]


## 11. Weighted ensemble models (soft voting)

The paper builds two custom weighted ensembles combining the favourable attributes of
the three models. Naive Bayes carries the highest weight because of its strength at
forecasting attrition. K-NN is wrapped in a scaling pipeline so the whole ensemble can
operate on a single feature matrix.

| | Ensemble 1 (all features) | Ensemble 2 (selected features) |
|---|---|---|
| Naive Bayes | weight 0.5 | weight 0.5 |
| K-NN (grid search) | weight 0.1 | weight 0.2 |
| Random Forest (grid search) | weight 0.3 | weight 0.3 |

In [30]:
# ----- Ensemble 1: all features  (NB 0.5 / KNN 0.1 / RF 0.3) -----
knn_best_all = GridSearchCV(KNeighborsClassifier(), knn_param_grid,
                            scoring="f1", cv=3, n_jobs=-1)
knn_best_all.fit(make_sets(list(X.columns), scale=True)[0], y_train)

ens1 = VotingClassifier(
    estimators=[
        ("nb", GaussianNB()),
        ("knn", Pipeline([("sc", StandardScaler()),
                          ("knn", KNeighborsClassifier(**knn_best_all.best_params_))])),
        ("rf", RandomForestClassifier(**rf_grid_all.best_params_)),
    ],
    voting="soft",
    weights=[0.5, 0.1, 0.3],
)
ens1.fit(X_train, y_train)
results[("Ensemble 1", "All")] = evaluate(ens1, X_test, y_test,
                                          "Ensemble 1 (NB+KNN+RF, all features)")

Ensemble 1 (NB+KNN+RF, all features)          Accuracy = 80.9524%  F1 = 51.7241%
   Confusion matrix: [[312, 58], [26, 45]]


In [31]:
# ----- Ensemble 2: selected features  (NB 0.5 / KNN 0.2 / RF 0.3) -----
ens2 = VotingClassifier(
    estimators=[
        ("nb", GaussianNB()),
        ("knn", Pipeline([("sc", StandardScaler()),
                          ("knn", KNeighborsClassifier(**knn_grid_sel.best_params_))])),
        ("rf", RandomForestClassifier(**rf_grid_sel.best_params_)),
    ],
    voting="soft",
    weights=[0.5, 0.2, 0.3],
)
ens2.fit(X_train[selected_features], y_train)
results[("Ensemble 2", "Selected")] = evaluate(
    ens2, X_test[selected_features], y_test, "Ensemble 2 (NB+KNN+RF, selected features)"
)

Ensemble 2 (NB+KNN+RF, selected features)     Accuracy = 84.5805%  F1 = 50.0000%
   Confusion matrix: [[339, 31], [37, 34]]


## 12. Results summary

Collected accuracy / F1 across all models and feature sets (the paper's Table II & III).
Because no random seed is fixed, exact values will vary per run but should track the
paper: RF strongest on overall accuracy, Naive Bayes strongest at flagging attrition,
and Ensemble 2 giving the best balanced accuracy (~83–84%).

In [32]:
summary = pd.DataFrame(
    [(m, fs, "{:.2%}".format(acc), "{:.2%}".format(f1))
     for (m, fs), (acc, f1) in results.items()],
    columns=["Model", "Feature set", "Accuracy", "F1 score"],
)
summary

,Model,Feature set,Accuracy,F1 score
0,RF,Selected,85.71%,30.77%
1,RF,New Selected,86.39%,34.78%
2,RF,Grid+Selected,86.39%,36.17%
3,RF,Grid+New Selected,86.62%,39.18%
4,KNN,Selected,83.67%,25.00%
5,KNN,New Selected,85.49%,31.91%
6,KNN,Grid+Selected,83.90%,29.70%
7,KNN,Grid+New Selected,85.71%,35.05%
8,NB,Selected,78.68%,50.53%
9,NB,New Selected,77.55%,48.17%
